In [ ]:
bids_folder = '/data/ds-tmsrisk/'
import os.path as op
import arviz as az

from tms_risk.behavior.fit_model import build_model, get_data
import pymc as pm
from tqdm.notebook import tqdm

In [ ]:
dfs = []
models = []
idatas = []

model_labels  = ['flexible2_null', 'flexible2', 'flexible2a', 'flexible2b', '11a', '11b', '11c', '11_null']

for model_label in tqdm(model_labels):
    dfs.append(get_data(model_label=model_label))

    model = build_model(model_label, dfs[-1])
    idata = az.from_netcdf(op.join(bids_folder, 'derivatives', 'cogmodels', f'model-{model_label}_trace.netcdf')).sel(draw=slice(None, None, 2))
    model.build_estimation_model()
    with model.estimation_model:
        pm.compute_log_likelihood(idata)

    models.append(build_model(model_label, dfs[-1]))
    idatas.append(idata)

In [ ]:
model_mapping = {
    'flexible2': 'Flexible PMC model (TMS affects both perception and working memory)',
    'flexible2a':'Flexible PMC model (TMS affects working memory only)',
    'flexible2b':'Flexible PMC model (TMS affects perception only)',
    '11a':'Weber PMC model (TMS affects memory only)',
    '11b':'Weber PMC model (TMS affects perception only)',
    '11c':'Weber PMC model (TMS affects both perception and memory)',
    '11_null':'Weber PMC null model',
    'flexible2_null':'Flexible PMC null model'}

In [ ]:
models = {}
for model_label, idata in zip(model_labels, idatas):
    models[model_mapping[model_label]] = idata

In [ ]:
comparison = az.compare(models)

In [ ]:
az.plot_compare(comparison)

In [ ]:
comparison

In [ ]:
import ace_tools as tools
tools.display_dataframe_to_user("Model Comparison Table", comparison)